<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a> 著作《<a href="http://mng.bz/orYv">从零构建大语言模型 (Build a Large Language Model From Scratch)</a>》配套补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


#第2章：处理文本数据

此笔记本中使用的程序包：

In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.5.1
tiktoken version: 0.7.0


-本章介绍数据准备和采样，以便为LLM “准备好”输入数据

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/01.webp?timestamp=1" width="500px">

# # 2.1了解词嵌入

- 本节没有相关代码。

-嵌入有多种形式；我们专注于本书中的文本嵌入

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/02.webp" width="500px">

- LLM使用高维空间（即数千维空间）中的嵌入
-由于我们无法想象这种高维空间（我们人类在1、2或3维空间中思考），因此下图说明了一个二维嵌入空间

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/03.webp" width="300px">

# # 2.2标记文本

-在本节中，我们将文本标记化，这意味着将文本分解为更小的单位，例如单个单词和标点符号

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/04.webp" width="300px">

-加载我们想要处理的原始文本
- [The Verdict by Edith Wharton] (https://en.wikisource.org/wiki/The_Verdict)是一部公共领域的短篇小说

In [2]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)


# The book originally used the following code below
# However, urllib uses older protocol settings that
# can cause problems for some readers using a VPN.
# The `requests` version above is more robust
# in that regard.

"""
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)
"""

<br>

---

<br>

#### 解决 SSL 证书错误问题

- 部分读者反馈在 VSCode 或 Jupyter 中运行 `urllib.request.urlretrieve` 时遇到了 ssl.SSLCertVerificationError: `SSL: CERTIFICATE_VERIFY_FAILED` 错误。
- 这通常意味着 Python 的证书包（certificate bundle）已过时。


**解决方案**

- 使用 Python ≥ 3.9；可以通过运行以下代码检查你的 Python 版本：
```python
import sys
print(sys.__version__)
```
- 升级证书包：
  - pip: `pip install --upgrade certifi`
  - uv: `uv pip install --upgrade certifi`
- 升级后请重启 Jupyter 内核。
- 如果在执行前面的代码单元格时仍然遇到 `ssl.SSLCertVerificationError`，请参阅 GitHub 上的相关讨论：[查看更多信息](https://github.com/rasbt/LLMs-from-scratch/pull/403)

<br>

---

<br>

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


-目标是为LLM标记并嵌入此文本
-让我们基于一些简单的示例文本开发一个简单的标记器，然后我们可以将其应用于上面的文本
-以下正则表达式将在空格上拆分

In [4]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


-我们不仅要对空格进行拆分，还要对逗号和句点进行拆分，因此让我们也修改正则表达式以执行此操作

In [5]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


-正如我们所看到的，这会创建空字符串，让我们删除它们

In [6]:
# Strip whitespace from each item and then filter out any empty strings.
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


-这看起来不错，但让我们也处理其他类型的标点符号，如句点、问号等

In [7]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


-这非常好，我们现在已经准备好将此标记化应用于原始文本

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/05.webp" width="350px">

In [8]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


-让我们计算代币总数

In [9]:
print(len(preprocessed))

4690


# # 2.3将令牌转换为令牌ID

-接下来，我们将文本令牌转换为令牌ID，稍后可以通过嵌入层进行处理

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/06.webp" width="500px">

-从这些代币中，我们现在可以构建一个由所有独特代币组成的词汇表

In [10]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [11]:
vocab = {token:integer for integer,token in enumerate(all_words)}

-以下是此词汇表中的前50个条目：

In [12]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


-下面，我们用一个小词汇说明了一个简短示例文本的标记化：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/07.webp?123" width="500px">

-现在把它们放在一个标记器类中

In [13]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

—`encode`功能将文本转换为token id
—`decode`功能将token id转换回文本

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="500px">

-我们可以使用标记器将文本编码（即标记）为整数
-这些整数可以嵌入（稍后）作为LLM的输入

In [14]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


-我们可以将整数解码回文本

In [15]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [16]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

,
添加特殊的上下文令牌

为未知单词添加一些“特殊”标记并表示文本的结尾是有用的

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="500px">

- 一些分词器会使用特殊标记（special token）来为大语言模型提供额外的上下文信息
- 其中一些特殊标记包括：
  - `[BOS]`（序列开头，beginning of sequence）用于标记文本的开头
  - `[EOS]`（序列结尾，end of sequence）用于标记文本的结束位置（通常用于拼接多段不相关的文本，例如两篇不同的维基百科文章或两本不同的书等）
  - `[PAD]`（填充，padding）：当我们以大于 1 的批次大小（batch size）训练 LLM 时（输入可能包含长度不同的多个文本；使用填充标记可以将较短的文本填充到与最长文本相同的长度，使所有文本长度一致）
- `[UNK]` 用于表示词汇表中未包含的未知词（unknown words）

- 请注意，GPT-2 并不需要上述提到的这些标记，它仅使用一个 `<|endoftext|>` 标记来降低复杂度
- `<|endoftext|>` 类似于上述的 `[EOS]` 标记
- GPT 也使用 `<|endoftext|>` 进行填充（由于在批次输入上进行训练时我们通常会使用掩码，因此不会对填充标记分配注意力，这些标记具体是什么并不重要）
- GPT-2 不会为词表外单词使用 `<UNK>` 标记；相反，GPT-2 使用字节对编码（BPE）分词器，将单词拆分为子词单元（subword units），我们将在后面的章节中对此进行讨论


-我们在两个独立的文本源之间使用`QQ0000QQ`token：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="500px">

-让我们看看如果将以下文本标记化会发生什么：

In [17]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

KeyError: 'Hello'

-由于词汇中不包含“Hello”一词，以上内容会产生错误
-为了处理此类情况，我们可以在词汇表中添加`"QQ0000QQ"`等特殊令牌来表示未知单词
-由于我们已经扩展了词汇表，让我们添加另一个名为`"QQ0001QQ"`的令牌，用于GPT-2培训以表示文本的结尾（并且它也用于连接文本之间，例如如果我们的培训数据集由多个文章，书籍等组成）

In [18]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [19]:
len(vocab.items())

1132

In [20]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


-我们还需要相应地调整令牌生成器，以便它知道何时以及如何使用新的`QQ0000QQ`token

In [21]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

让我们尝试使用修改后的标记器对文本进行标记：

In [22]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [23]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [24]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

# # 2.5字节对编码

- GPT-2 使用字节对编码（BytePair encoding, BPE）作为其分词器
- 它允许模型将不在预定义词汇表中的单词分解为更小的子词单元甚至单个字符，从而使其能够处理词表外（out-of-vocabulary）单词
- 例如，如果 GPT-2 的词汇表中没有 "unfamiliarword" 这个词，它可能会根据其训练好的 BPE 合并规则将其分词为 ["unfam", "iliar", "word"] 或其他子词划分
- 原始的 BPE 分词器代码可以在这里找到：[https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- 在本章中，我们使用 OpenAI 开源的 [tiktoken](https://github.com/openai/tiktoken) 库中的 BPE 分词器，该库使用 Rust 实现了其核心算法以提升计算性能
- 我在 [./bytepair_encoder](../02_bonus_bytepair-encoder) 中创建了一个笔记本，对这两种实现进行了对比（tiktoken 在示例文本上的运行速度大约快了 5 倍）

In [25]:
# pip install tiktoken

In [26]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.7.0


In [27]:
tokenizer = tiktoken.get_encoding("gpt2")

In [28]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [29]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


- BPE标记器将未知单词分解为子单词和单个字符：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="300px">

# # 2.6使用滑动窗口进行数据采样

-我们训练LLM一次生成一个单词，因此我们希望相应地准备训练数据，其中序列中的下一个单词表示要预测的目标：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="400px">

In [30]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


-对于每个文本块，我们需要输入和目标
-由于我们希望模型预测下一个单词，因此目标是向右移动一个位置的输入

In [31]:
enc_sample = enc_text[50:]

In [32]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


-一个接一个的预测如下所示：

In [33]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [34]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


-在我们讨论了注意力机制之后，我们将在后面的章节中处理下一个单词的预测
-目前，我们实现了一个简单的数据加载器，它对输入数据集进行迭代，并返回偏移1的输入和目标

-安装和导入PyTorch （有关安装提示，请参阅附录A）

In [35]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.5.1


-我们使用滑动窗口方法，将位置更改+1：<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/13.webp?123" width="500px">

-创建数据集和数据加载器，从输入文本数据集中提取块

In [36]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [37]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

-让我们为上下文大小为4的LLM测试批量大小为1的数据加载器：

In [38]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [39]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [40]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


-使用步幅等于上下文长度（此处为： 4）的示例，如下所示：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="500px">

-我们还可以创建批量输出
-请注意，我们在此处增加步幅，以便批次之间没有重叠，因为更多的重叠可能导致过拟合增加

In [41]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


# # 2.7创建令牌嵌入

-数据几乎已经为LLM做好了准备
-但最后让我们使用嵌入层将令牌嵌入到连续矢量表示中
-通常，这些嵌入层是LLM本身的一部分，并在模型培训期间更新（训练）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="400px">

-假设我们有以下四个输入示例，输入ID为2、3、5和1 （标记化后）：

In [42]:
input_ids = torch.tensor([2, 3, 5, 1])

-为了简单起见，假设我们只有6个单词的小词汇表，我们想创建大小为3的嵌入：

In [43]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

-这将导致6x3权重矩阵：

In [44]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


- 对于熟悉独热编码（one-hot encoding）的读者来说，上述嵌入层的方法本质上只是一种更高效的实现方式，其效果等同于全连接层中的独热编码后紧接着矩阵乘法，这在 [./embedding_vs_matmul](../03_bonus_embedding-vs-matmul) 的补充代码中有详细介绍
- 因为嵌入层只是一种更高效的等效实现（等同于独热编码加矩阵乘法的方法），所以它可以被看作是一个可以通过反向传播进行优化的神经网络层

-要将id为3的令牌转换为三维向量，我们执行以下操作：

In [45]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


-请注意，以上是`embedding_layer`权重矩阵中的第4行
-要嵌入上述所有四个`input_ids`值，我们需要

In [46]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


-嵌入层本质上是一种查找操作：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="500px">

- * *您可能对将嵌入层与常规线性层进行比较的奖励内容感兴趣： [../03_bonus_embedding-vs-matmul] (../03_bonus_embedding-vs-matmul) * *

# # 2.8编码单词位置

-嵌入层将ID转换为相同的矢量表示，无论它们在输入序列中的位置如何：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/17.webp" width="400px">

-位置嵌入与标记嵌入向量相结合，形成大型语言模型的输入嵌入：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="500px">

- BytePair编码器的词汇量为50,257：
假设我们想要将输入符号编码为256维向量表示：

In [47]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

-如果我们从数据加载器中采样数据，我们将每批中的令牌嵌入到256维向量中
-如果我们的批处理大小为8，每个令牌4个，这将产生8 x 4 x 256张量：

In [48]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [49]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [50]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(token_embeddings)

torch.Size([8, 4, 256])


GPT-2使用绝对位置嵌入，所以我们只需要创建另一个嵌入层：

In [51]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# uncomment & execute the following line to see how the embedding layer weights look like
# print(pos_embedding_layer.weight)

In [52]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(pos_embeddings)

torch.Size([4, 256])


为了创建LLM中使用的输入嵌入，我们只需添加token和位置嵌入：

In [53]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(input_embeddings)

torch.Size([8, 4, 256])


—在输入处理工作流的初始阶段，输入文本被分割成单独的令牌
-在此分割之后，这些令牌根据预定义的词汇表转换为令牌id：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/19.webp" width="400px">

&nbsp;
## 总结与要点

请参阅[./dataloader.ipynb] (./dataloader.ipynb)代码Notebook，这是我们在本章中实现的数据加载器的简洁版本，需要在即将到来的章节中训练GPT模型。

有关锻炼解决方案，请参阅[./exercise-solutions.ipynb] (./exercise-solutions.ipynb)。

如果您有兴趣了解如何从头开始实现和训练GPT-2标记器，请参阅[字节对编码（BPE）标记器从头开始] （../02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb）Notebook。